In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Cargar datos procesados
df = pd.read_csv('../data/processed/siata_features.csv')

# Definir features y variable objetivo
FEATURES = ['hora', 'dia_semana', 'mes', 'es_fin_semana', 
            'pm25_lag1', 'pm25_lag3', 'pm25_lag6', 'pm25_lag24',
            'pm25_media3h', 'pm25_media6h', 'estacion']

TARGET = 'pm25_siguiente'

X = df[FEATURES]
y = df[TARGET]

# Dividir en entrenamiento y prueba (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)

Datos de entrenamiento: (146764, 11)
Datos de prueba: (36692, 11)


In [3]:
# Conectar con MLflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("prediccion-pm25-siata")

def evaluar_modelo(y_true, y_pred):
    """Calcula las métricas de evaluación"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {"mae": mae, "rmse": rmse, "r2": r2}

# ── Experimento 1: Regresión Lineal ──
print("Entrenando Regresión Lineal...")
with mlflow.start_run(run_name="regresion-lineal"):
    
    # Parámetros
    params = {"modelo": "LinearRegression"}
    mlflow.log_params(params)
    
    # Entrenar
    modelo_lr = LinearRegression()
    modelo_lr.fit(X_train, y_train)
    
    # Evaluar
    y_pred = modelo_lr.predict(X_test)
    metricas = evaluar_modelo(y_test, y_pred)
    mlflow.log_metrics(metricas)
    
    # Guardar modelo
    mlflow.sklearn.log_model(modelo_lr, "modelo")
    
    print(f"  MAE:  {metricas['mae']:.2f}")
    print(f"  RMSE: {metricas['rmse']:.2f}")
    print(f"  R2:   {metricas['r2']:.3f}")

print("\n Experimento 1 registrado en MLflow")

2026/05/02 13:41:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Entrenando Regresión Lineal...


2026/05/02 13:41:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/02 13:41:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


  MAE:  6.12
  RMSE: 8.94
  R2:   0.511
🏃 View run regresion-lineal at: http://localhost:5000/#/experiments/1/runs/1d794a3115ed49baab85ca80d955b13c
🧪 View experiment at: http://localhost:5000/#/experiments/1

 Experimento 1 registrado en MLflow


In [5]:
# ── Experimento 2: Random Forest ──
print("Entrenando Random Forest...")
with mlflow.start_run(run_name="random-forest-base"):
    
    params = {
        "modelo": "RandomForestRegressor",
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 42
    }
    mlflow.log_params(params)
    
    modelo_rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    modelo_rf.fit(X_train, y_train)
    
    y_pred = modelo_rf.predict(X_test)
    metricas = evaluar_modelo(y_test, y_pred)
    mlflow.log_metrics(metricas)
    
    mlflow.sklearn.log_model(modelo_rf, "modelo")
    
    print(f"  MAE:  {metricas['mae']:.2f}")
    print(f"  RMSE: {metricas['rmse']:.2f}")
    print(f"  R2:   {metricas['r2']:.3f}")

print("\n Experimento 2 registrado en MLflow")

Entrenando Random Forest...


2026/05/02 13:43:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 13:43:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/02 13:43:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


  MAE:  5.59
  RMSE: 8.11
  R2:   0.597
🏃 View run random-forest-base at: http://localhost:5000/#/experiments/1/runs/bb7dcc95820d4676b56740d22d8d0d92
🧪 View experiment at: http://localhost:5000/#/experiments/1

 Experimento 2 registrado en MLflow


In [6]:
# ── Experimento 3: Random Forest optimizado ──
print("Entrenando Random Forest optimizado...")
with mlflow.start_run(run_name="random-forest-optimizado"):
    
    params = {
        "modelo": "RandomForestRegressor",
        "n_estimators": 200,
        "max_depth": 15,
        "min_samples_split": 5,
        "random_state": 42
    }
    mlflow.log_params(params)
    
    modelo_rf2 = RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )
    modelo_rf2.fit(X_train, y_train)
    
    y_pred = modelo_rf2.predict(X_test)
    metricas = evaluar_modelo(y_test, y_pred)
    mlflow.log_metrics(metricas)
    
    mlflow.sklearn.log_model(modelo_rf2, "modelo")
    
    # Guardar importancia de features
    importancias = pd.DataFrame({
        'feature': FEATURES,
        'importancia': modelo_rf2.feature_importances_
    }).sort_values('importancia', ascending=False)
    
    print(f"  MAE:  {metricas['mae']:.2f}")
    print(f"  RMSE: {metricas['rmse']:.2f}")
    print(f"  R2:   {metricas['r2']:.3f}")
    print(f"\nFeatures más importantes:")
    print(importancias.to_string(index=False))

print("\n✅ Experimento 3 registrado en MLflow")

Entrenando Random Forest optimizado...


2026/05/02 13:45:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 13:45:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/02 13:45:32 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


  MAE:  5.49
  RMSE: 8.00
  R2:   0.608

Features más importantes:
      feature  importancia
    pm25_lag1     0.652446
         hora     0.070770
   pm25_lag24     0.067928
 pm25_media3h     0.039392
 pm25_media6h     0.037720
    pm25_lag6     0.034394
    pm25_lag3     0.031166
          mes     0.025087
     estacion     0.024004
   dia_semana     0.015408
es_fin_semana     0.001686
🏃 View run random-forest-optimizado at: http://localhost:5000/#/experiments/1/runs/3877e399187b447087605b186f6ec515
🧪 View experiment at: http://localhost:5000/#/experiments/1

✅ Experimento 3 registrado en MLflow


In [7]:
# Registrar el mejor modelo en el Model Registry
run_id = "3877e399187b447087605b186f6ec515"  # el ID del experimento 3

model_uri = f"runs:/{run_id}/modelo"

mlflow.register_model(
    model_uri=model_uri,
    name="pm25-predictor"
)

print(" Modelo registrado en MLflow Model Registry")
print(f"   Nombre: pm25-predictor")
print(f"   URI: {model_uri}")

Successfully registered model 'pm25-predictor'.
2026/05/02 13:49:43 WARNING mlflow.tracking._model_registry.fluent: Run with id 3877e399187b447087605b186f6ec515 has no artifacts at artifact path 'modelo', registering model based on models:/m-0ade34a865da49ab8184eb2f915cc689 instead
2026/05/02 13:49:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: pm25-predictor, version 1


 Modelo registrado en MLflow Model Registry
   Nombre: pm25-predictor
   URI: runs:/3877e399187b447087605b186f6ec515/modelo


Created version '1' of model 'pm25-predictor'.
